# Multi-Head Attention (MHA)

## 1. Purpose

Multi-Head Attention allows the model to attend to different parts of a sequence using multiple independent attention heads.

Instead of performing one attention operation over the entire `d_model`-dimensional representation, we split the representation into several smaller heads.

For example:

```text
d_model = 16
num_heads = 4

head_dim = d_model / num_heads
        = 16 / 4
        = 4
```

So each token's 16 features are divided across 4 attention heads, with 4 features per head.

---

## 2. Input Shape

For a batch of sequences:

```text
[batch_size, sequence_length, d_model]
```

Example:

```text
[4, 8, 16]
```

Meaning:

* 4 sequences in the batch
* 8 tokens per sequence
* 16 features per token

---

## 3. Q, K, V Projections

The input is passed through three learned linear layers:

```python
Q = W_q(x)
K = W_k(x)
V = W_v(x)
```

If:

```python
x.shape = [4, 8, 16]
```

and each linear layer is:

```python
nn.Linear(16, 16)
```

then:

```text
Q.shape = [4, 8, 16]
K.shape = [4, 8, 16]
V.shape = [4, 8, 16]
```

---

## 4. Splitting Into Heads

The 16-dimensional representation is divided into 4 heads:

```text
[batch, sequence, d_model]
          ↓
[batch, sequence, num_heads, head_dim]
```

Therefore:

```text
[4, 8, 16]
      ↓
[4, 8, 4, 4]
```

because:

```text
16 = 4 × 4
```

The `.view()` operation only reorganizes the tensor. It does not perform a mathematical transformation.

---

## 5. Transposing

We then move the head dimension before the sequence dimension:

```text
[batch, sequence, heads, head_dim]
              ↓
[batch, heads, sequence, head_dim]
```

Therefore:

```text
[4, 8, 4, 4]
      ↓
[4, 4, 8, 4]
```

This allows attention to be calculated independently for every head.

---

## 6. Scaled Dot-Product Attention

For each head:

```text
Attention(Q,K,V)
=
softmax(QKᵀ / √d_k)V
```

First:

```text
Q.shape = [4, 4, 8, 4]

K.shape = [4, 4, 8, 4]
```

Transpose the last two dimensions of K:

```text
Kᵀ.shape = [4, 4, 4, 8]
```

Then:

```text
Q @ Kᵀ

[4, 4, 8, 4]
        ×
[4, 4, 4, 8]

        ↓

[4, 4, 8, 8]
```

The final two dimensions form an `8 × 8` attention matrix.

This means that for every token, the model calculates how much attention it should give to every token in the sequence.

---

## 7. Scaling

The attention scores are divided by:

```text
√d_k
```

where `d_k` is the dimensionality of one head.

For our example:

```text
d_k = 4
```

so:

```text
scores / √4
```

Scaling prevents the dot products from becoming too large and causing the softmax to become excessively peaked.

---

## 8. Softmax

```python
attention_weights = torch.softmax(scores, dim=-1)
```

The softmax is applied across the key/token dimension.

Therefore, for each query token, the attention weights across all tokens sum to approximately 1.

---

## 9. Weighted Sum of V

The attention weights are multiplied by V:

```text
attention_weights @ V
```

Shapes:

```text
[4, 4, 8, 8]
        ×
[4, 4, 8, 4]

        ↓

[4, 4, 8, 4]
```

Each token now has a new representation containing information gathered from other tokens.

---

## 10. Combining the Heads

After attention:

```text
[batch, heads, sequence, head_dim]

[4, 4, 8, 4]
```

Transpose again:

```text
[4, 8, 4, 4]
```

Then concatenate the heads:

```text
[4, 8, 4, 4]
        ↓
[4, 8, 16]
```

because:

```text
num_heads × head_dim
=
4 × 4
=
16
```

We have reconstructed the original `d_model` dimension.

---

## 11. Final Linear Projection

The concatenated output is passed through another learned linear layer:

```python
self.fc_out = nn.Linear(d_model, d_model)
```

Therefore:

```text
[4, 8, 16]
      ↓
Linear(16 → 16)
      ↓
[4, 8, 16]
```

The output has the same shape as the input.

---

## 12. Complete Shape Flow

For:

```text
batch_size = 4
sequence_length = 8
d_model = 16
num_heads = 4
head_dim = 4
```

the complete shape flow is:

```text
Input
[4, 8, 16]

      ↓ Q/K/V projections

[4, 8, 16]

      ↓ split into heads

[4, 8, 4, 4]

      ↓ transpose

[4, 4, 8, 4]

      ↓ Q @ Kᵀ

[4, 4, 8, 8]

      ↓ softmax

[4, 4, 8, 8]

      ↓ @ V

[4, 4, 8, 4]

      ↓ transpose

[4, 8, 4, 4]

      ↓ concatenate heads

[4, 8, 16]

      ↓ final linear layer

[4, 8, 16]
```

## 13. The Main Idea

The complicated-looking shape operations are doing only one major thing:

```text
16 features
    ↓
4 heads × 4 features
    ↓
attention independently in each head
    ↓
4 heads × 4 features
    ↓
16 features again
```

The `.view()` and `.transpose()` operations are primarily **reorganizing dimensions so that attention can be performed independently across heads**.

## 14. Important Constraint

`d_model` must be divisible by `num_heads`:

```text
d_model % num_heads == 0
```

For example:

```text
d_model = 16
num_heads = 4   ✓

d_model = 32
num_heads = 8   ✓

d_model = 16
num_heads = 3   ✗
```

because each head must receive an equal number of features.

## 15. Encoder Self-Attention

For an encoder-only Transformer:

```python
self_attention(x, x, x)
```

is used.

Therefore:

```text
Q = x
K = x
V = x
```

after their respective learned projections.

Each token can therefore attend to other tokens in the input sequence.

## 16. Padding Mask

When working with real text, sequences usually have different lengths.

For example:

```text
"I like this movie"
"I really like this movie a lot"
```

They may need to be padded to the same length in a batch.

The model should not treat padding tokens as meaningful words.

A padding mask tells attention:

```text
"Do not attend to these padding positions."
```

This becomes important when moving from random tensors to the DAIC-WOZ dataset.


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import math

In [3]:
vocab_size = 20
embedding_dim = 16
batch_size = 4
sequence_length = 8
x = torch.randint(0, vocab_size, (4,5))
y = torch.randint(0, 2, (4,))

In [4]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model=16, max_len=50):
        super(PositionalEncoding, self).__init__()

        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2)
            * (-math.log(10000.0) / d_model)
        )

        pe = torch.zeros(max_len, d_model)

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

In [5]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model = 16, num_heads = 4):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "Embedding dimension must be divisible by the number of heads."

        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # Linear layers for Q, K, V transformations
        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.fc_out = nn.Linear(d_model, d_model)

    def forward(self, query, key, value):
        batch_size = query.size(0)

        # Perform linear transformations and split into heads
        Q = self.q_linear(query).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.k_linear(key).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.v_linear(value).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        # Compute attention
        attn_weights = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn_weights = torch.softmax(attn_weights, dim=-1)
        attn_output = torch.matmul(attn_weights, V)

        # Concatenate heads and put through final linear layer
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, -1, self.num_heads * self.d_k)
        return self.fc_out(attn_output)

In [6]:
class FeedForwardNetwork(nn.Module):
    def __init__(self, d_model = 16, d_ff = 4):
        super(FeedForwardNetwork, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(0.1)
        self.activation = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.dropout(self.activation(self.fc1(x))))

In [9]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_ff = 4, dropout=0.1, d_model = 16, max_len = 50, num_heads = 4):
        super(TransformerEncoderLayer, self).__init__()
        self.we = nn.Embedding(vocab_size, embedding_dim)
        self.pe = PositionalEncoding(d_model = d_model, max_len = max_len)
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForwardNetwork(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, 2)

    def forward(self, x):

        x = self.we(x)
        x = self.pe(x)
        # Self-Attention with Residual Connection
        x = x + self.dropout(self.self_attn(x, x, x))
        x = self.norm1(x)

        # Feedforward with Residual Connection
        x = x + self.dropout(self.ffn(x))
        x = self.norm2(x)
        last_output = x[:, -1, :]
        logits = self.classifier(last_output)

        return logits


In [10]:
model = TransformerEncoderLayer()
optimizer = optim.SGD(model.parameters(), lr=0.1)
criterion = nn.CrossEntropyLoss()
epochs = 10
for epoch in range(epochs):
    logits = model(x)
    loss = criterion(logits, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

Epoch [1/10], Loss: 0.7210
Epoch [2/10], Loss: 0.5015
Epoch [3/10], Loss: 0.3968
Epoch [4/10], Loss: 0.3485
Epoch [5/10], Loss: 0.2636
Epoch [6/10], Loss: 0.2094
Epoch [7/10], Loss: 0.1748
Epoch [8/10], Loss: 0.1603
Epoch [9/10], Loss: 0.1246
Epoch [10/10], Loss: 0.1258
